# Download the priors

Downloads PIE's five genetic knowledge sources into `priors/raw/<source>/` under the data root, plus the two text sources that only PIE uses: `context_text` (the cell context's description) and `gene_text` (the readout genes' descriptions).
Each gives every gene a vector: text embeddings of the gene's perturbation description and of its NCBI annotations, an ESM2 protein embedding, a STRING network embedding (SPACE), and DepMap gene effects (Chronos).
The files are PIE's own release on Hugging Face (`arcinstitute/PIE_sources`), at the commit PIE's Replogle–Nadig config pins, in PIE's source format (`meta.json`, `embeddings.npy`, and for some sources `offsets.npy`, `aliases.yaml`, `descriptions.json`).
The download is 43 GB, almost all of it `ncbi_text`, which keeps one vector per token.
Existing files of the right size are skipped, and every large file is checked against its sha256.

In [ ]:
import hashlib
import json
import sys
import urllib.request

import pandas as pd

sys.path.append("../utils")
from download import download
from paths import data_dir

In [ ]:
SOURCES = ["perturbation_text", "ncbi_text", "esm2", "string_space", "depmap_gene_effect"]
PIE_ONLY = ["context_text", "gene_text"]  # PIE reads these too (2.train_test/1.train_pie.ipynb); PWPA does not
REV = "fb624a5158a7f37dfa13a015cdacadb985549d26"  # pinned in PIE's configs/experiment/replogle_wdataset.yaml (PIE @ 21a526a)
HF = "https://huggingface.co/datasets/arcinstitute/PIE_sources"


def get(url):
    return urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "agent-local"}), timeout=120)


tree = json.load(get(f"https://huggingface.co/api/datasets/arcinstitute/PIE_sources/tree/{REV}?recursive=true"))
files = [f for f in tree if f["type"] == "file" and f["path"].split("/")[0] in SOURCES + PIE_ONLY]
for f in files:
    dst = data_dir("priors", "raw") / f["path"]  # <source>/<file>
    sha, size = f.get("lfs", {}).get("oid"), f.get("lfs", {}).get("size", f["size"])  # small files are not in LFS
    if dst.exists() and dst.stat().st_size == size:
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    download(f"{HF}/resolve/{REV}/{f['path']}", dst, f"sha256:{sha}" if sha else "")
    assert dst.stat().st_size == size, (f["path"], dst.stat().st_size, size)
    print("got", f["path"], size)
print(len(files), "files in", data_dir("priors", "raw"))

The T-cell CRISPR screens (BioGRID ORCS and two papers' supplementary tables, the `tcell_screens` row of [`datasets.tsv`](datasets.tsv)) go into `priors/raw/tcell_screens/`, for [`3.further_analysis`](../3.further_analysis/).

In [ ]:
row = pd.read_csv("datasets.tsv", sep="\t").set_index("dataset").loc["tcell_screens"]
out = data_dir("priors", "raw", "tcell_screens")
for name, url, checksum in zip(row.file.split(), row.url.split(), row.checksum.split()):
    path = out / name
    algo, _, digest = checksum.partition(":")
    if path.exists():
        with open(path, "rb") as f:
            if hashlib.file_digest(f, algo).hexdigest() == digest:
                continue
    download(url, path, checksum)
    print("got", name, path.stat().st_size)